# 🎯 Module 07: CNN — Convolutional Neural Networks

> **Stage:** Deep Learning  
> **Module:** Convolutional Neural Networks (CNNs)  
> **Framework:** PyTorch  
> **Date:** 07 June 2026

---

Convolutional Neural Networks are the workhorse of modern computer vision.
In this module we will build CNNs **from the ground up** — starting with a single convolution
and ending with a trained image classifier on a real dataset.

We will cover:

- Why CNNs beat dense networks for images
- Image tensors & channels
- Convolution, kernels, feature maps
- Stride, padding, output size math
- Receptive fields
- Pooling (Max / Average)
- Building CNNs in PyTorch
- Training an image classifier end-to-end
- Debugging common CNN mistakes

## 📚 Learning Objectives

By the end of this notebook, you should be able to:

- Explain why CNNs are more suitable than dense networks for images
- Understand image tensor shapes `[N, C, H, W]`
- Perform a convolution manually and interpret feature maps
- Compute convolution output size and parameter counts
- Explain stride, padding, and receptive fields
- Use `nn.Conv2d`, `nn.MaxPool2d`, `nn.AdaptiveAvgPool2d`
- Build, train, and evaluate a CNN in PyTorch
- Diagnose common CNN bugs (wrong shape, wrong channels, etc.)

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is a CNN?

A **Convolutional Neural Network (CNN)** is designed for data with **spatial structure** — images, video, spectrograms, etc.

Instead of treating every pixel independently, CNNs learn **local patterns** and combine them hierarchically:

```text
Edges
 ↓
Textures
 ↓
Shapes
 ↓
Objects
```

### Typical CNN Pipeline

```text
Image
 ↓
Convolution → Activation → Pooling
 ↓
Convolution → Activation → Pooling
 ↓
Classifier (Linear layers)
 ↓
Prediction
```

---
## ❓ 2. Why Not a Normal Dense Network?

Consider a 224×224 RGB image → 224 × 224 × 3 = **150,528** input features.

A single dense layer with 1,000 neurons:

$$150{,}528 \times 1{,}000 = 150{,}528{,}000 \text{ weights}$$

That's **over 150 million weights in ONE layer** — before any bias!

CNNs solve this with:

| Trick | Benefit |
| --- | --- |
| Local connectivity | Each neuron sees only a small region |
| Parameter sharing | Same filter reused across the image |
| Hierarchy | Deep layers combine simple features |

Result: drastically fewer parameters while preserving spatial structure.

In [ ]:
# Parameter comparison: dense vs conv for a 224x224 RGB image
img = torch.randn(1, 3, 224, 224)

dense = nn.Linear(3 * 224 * 224, 1000)
conv  = nn.Conv2d(3, 64, kernel_size=3, padding=1)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'Dense  Linear(150528 → 1000): {count_params(dense):,} params')
print(f'Conv2d(3 → 64, k=3):         {count_params(conv):,} params')
print(f'\nConv uses {count_params(dense) / count_params(conv):,.0f}x fewer parameters!')

---
## 🖼️ 3. Image Tensors: `[N, C, H, W]`

PyTorch expects image batches in **channels-first** layout:

$$[N, C, H, W]$$

| Dim | Meaning |
| --- | --- |
| N | batch size |
| C | channels |
| H | height |
| W | width |

Examples:
- Grayscale batch: `[N, 1, H, W]`
- RGB batch: `[N, 3, H, W]`

⚠️ A very common bug is feeding `[N, H, W, C]` (TensorFlow/NumPy style) — it will silently produce garbage in PyTorch.

In [ ]:
# Inspect a fake image batch
batch = torch.randn(32, 3, 224, 224)
print('Batch shape:', batch.shape)
print('N =', batch.shape[0])
print('C =', batch.shape[1])
print('H =', batch.shape[2])
print('W =', batch.shape[3])

---
## 🧩 4. What Is Convolution?

A convolution slides a small matrix — called a **kernel** or **filter** — over the input,
performing element-wise multiplication and summing the result.

Example kernel (vertical edge detector):

$$K = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}$$

### Manual Example

Input (3×3):

$$X = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{bmatrix}$$

Kernel (2×2):

$$K = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$

Top-left region:

$$(1)(1) + (2)(0) + (4)(0) + (5)(1) = 1 + 5 = 6$$

The kernel then slides and repeats. The output is a **feature map**.

In [ ]:
# Implement a simple 2D convolution from scratch
def conv2d_manual(x, kernel, stride=1):
    """x: 2D tensor, kernel: 2D tensor, returns feature map."""
    Kh, Kw = kernel.shape
    H, W = x.shape
    H_out = (H - Kh) // stride + 1
    W_out = (W - Kw) // stride + 1
    out = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            region = x[i*stride:i*stride+Kh, j*stride:j*stride+Kw]
            out[i, j] = (region * kernel).sum()
    return out

X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.],
                  [7., 8., 9.]])
K = torch.tensor([[1., 0.],
                  [0., 1.]])

print('Input:\n', X)
print('Kernel:\n', K)
print('Feature map:\n', conv2d_manual(X, K))

---
## 🎨 5. What Does a Filter Learn?

The filter values are **learned during training** — we don't hand-design them.

```text
Early layers  →  Edges, corners, simple textures
Middle layers →  Shapes, parts
Deep layers   →  Objects, semantic concepts
```

This **hierarchical feature learning** is what makes CNNs so effective.

---
## 🔢 6. Channels & `nn.Conv2d`

Every filter operates over **all input channels** and produces **one output channel**.

| Input | Output | Meaning |
| --- | --- | --- |
| 3 ch | 64 filters | Output = 64 channels |

```python
nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3)
```

Input:  `[N, 3, H, W]`
Output: `[N, 64, H_out, W_out]`

### Kernel Sizes

Common choices: 3×3, 5×5, 7×7.
Smaller kernels stacked deep give large receptive fields with **fewer parameters** than one big kernel.

In [ ]:
# What a Conv2d layer actually does to shapes
x = torch.randn(8, 3, 32, 32)          # batch of 8 RGB 32x32 images
conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)

y = conv(x)
print('Input shape :', x.shape)
print('Output shape:', y.shape)
print('\nWeight shape:', conv.weight.shape, '(out_ch, in_ch, kH, kW)')
print('Bias shape  :', conv.bias.shape)

---
## 📐 7. Stride, Padding & Output Size

### Stride
How far the kernel moves each step:
- stride = 1 → move 1 pixel
- stride = 2 → move 2 pixels → output shrinks ~2×

### Padding
Adding a border of zeros around the input.
`padding = (K-1)/2` for stride 1 preserves spatial size.

### Output Size Formula

$$H_{out} = \left\lfloor \frac{H + 2P - K}{S} + 1 \right\rfloor$$

where H = input size, P = padding, K = kernel size, S = stride.

### Example 1 — Preserve size (32 → 32)
H=32, K=3, P=1, S=1:
$$H_{out} = \frac{32 + 2 - 3}{1} + 1 = 32$$

### Example 2 — Halve size (32 → 16)
H=32, K=3, P=1, S=2:
$$H_{out} = \left\lfloor \frac{32 + 2 - 3}{2} + 1 \right\rfloor = 16$$

In [ ]:
def conv_output_size(H, K, P=0, S=1):
    return (H + 2 * P - K) // S + 1

configs = [
    ('K=3, P=1, S=1', 32, 3, 1, 1),
    ('K=3, P=1, S=2', 32, 3, 1, 2),
    ('K=5, P=2, S=1', 32, 5, 2, 1),
    ('K=5, P=0, S=1', 32, 5, 0, 1),
    ('K=3, P=0, S=1', 32, 3, 0, 1),
]

print(f'{"Config":<18} {"H_in":>5} → {"H_out":>6}')
print('-' * 32)
for label, H, K, P, S in configs:
    print(f'{label:<18} {H:>5} → {conv_output_size(H, K, P, S):>6}')

# Verify against PyTorch
x = torch.randn(1, 1, 32, 32)
print('\nPyTorch check:')
for label, H, K, P, S in configs:
    y = nn.Conv2d(1, 1, K, stride=S, padding=P)(x)
    print(f'{label:<18} → {tuple(y.shape[2:])}')

---
## 🧮 8. Parameter Count for Conv2d

$$\text{Params} = C_{out} \times C_{in} \times K_h \times K_w + C_{out}$$

The `+ C_out` term is one bias per output channel.

### Example: 3 → 16 channels, 3×3 kernel

$$16 \times 3 \times 3 \times 3 + 16 = 432 + 16 = 448 \text{ params}$$

That's tiny compared to a fully-connected layer over the same input!

In [ ]:
def conv_params(in_ch, out_ch, K, bias=True):
    return out_ch * in_ch * K * K + (out_ch if bias else 0)

print('3 → 16, k=3, bias=True :', conv_params(3, 16, 3), 'params')
print('3 → 64, k=3, bias=True :', conv_params(3, 64, 3), 'params')
print('64 → 128, k=3, bias=True:', conv_params(64, 128, 3), 'params')

# Verify with PyTorch
c = nn.Conv2d(3, 16, kernel_size=3)
actual = sum(p.numel() for p in c.parameters())
print(f'\nPyTorch actual for 3→16 k=3: {actual} params')

---
## 🔗 9. Local Connectivity, Parameter Sharing & Receptive Fields

### Local Connectivity
Each filter only sees a small local region at a time:

```text
Image
┌──────────────┐
│   ┌──────┐   │
│   │Kernel│   │
│   └──────┘   │
└──────────────┘
```

### Parameter Sharing
The same filter weights are reused at every position:

```text
Position 1 → same filter
Position 2 → same filter
Position 3 → same filter
```

This gives **translation invariance** — a cat is still a cat whether it's on the left or right.

### Receptive Field
The region of the original input that influences a given neuron.

```text
Layer 1 → small receptive field
Layer 2 → larger
Layer 3 → even larger
```

Pooling and stride further expand the receptive field.

---
## 🌊 10. Pooling

Pooling reduces spatial dimensions:

**Max Pooling** — takes the maximum of a window:

$$\begin{bmatrix} 1 & 5 \\ 3 & 2 \end{bmatrix} \rightarrow 5$$

**Average Pooling** — takes the mean:

$$\frac{1 + 5 + 3 + 2}{4} = 2.75$$

```python
nn.MaxPool2d(2)   # 2×2 window, stride defaults to kernel_size
nn.AvgPool2d(2)
```

### Why Pooling?
- Reduces spatial dimensions → less computation
- Increases effective receptive field
- Adds some tolerance to small spatial shifts

In [ ]:
# Max vs Average pooling demo
x = torch.tensor([[[[1., 5.],
                    [3., 2.]]]])

max_pool = nn.MaxPool2d(2)
avg_pool = nn.AvgPool2d(2)

print('Input:\n', x.squeeze())
print('MaxPool2d(2):', max_pool(x).item())
print('AvgPool2d(2):', avg_pool(x).item())

# Effect on larger tensor
big = torch.randn(1, 64, 64, 64)
print('\n64×64 → MaxPool(2):', nn.MaxPool2d(2)(big).shape[2:])

---
## 🏗️ 11. Typical CNN Architecture

```text
Input Image
     ↓
Conv2d → ReLU → MaxPool
     ↓
Conv2d → ReLU → MaxPool
     ↓
Flatten
     ↓
Linear
     ↓
Output
```

Early layers learn simple patterns → deeper layers learn complex ones.

### Shape Trace (224×224 input)

```text
[32, 3, 224, 224]
   → Conv2d(3→32)      → [32, 32, 224, 224]
   → MaxPool(2)         → [32, 32, 112, 112]
   → Conv2d(32→64)      → [32, 64, 112, 112]
   → MaxPool(2)         → [32, 64, 56, 56]
   → Flatten            → [32, 200704]
   → Linear(200704→10)  → [32, 10]
```

In [ ]:
# Verify the shape trace
x = torch.randn(2, 3, 224, 224)

seq = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)

with torch.no_grad():
    for layer in seq:
        x = layer(x)
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            print(f'{layer.__class__.__name__:<12} → {tuple(x.shape)}')

---
## 🧰 12. Better Practice: `AdaptiveAvgPool2d`

Instead of hardcoding the flattened size (which breaks the moment you change the input size or architecture),
use **adaptive pooling** to force the output to a fixed spatial size:

```python
nn.AdaptiveAvgPool2d((1, 1))
```

```text
[N, 64, H, W]
     ↓
[N, 64, 1, 1]
```

Then `Flatten` → `[N, 64]` regardless of H and W. This is the pattern used in modern architectures (ResNet, etc.).

In [ ]:
# Adaptive pooling works for ANY spatial size
for size in [224, 128, 64, 32]:
    x = torch.randn(1, 64, size, size)
    y = nn.AdaptiveAvgPool2d((1, 1))(x)
    print(f'Input {size}×{size} → AdaptiveAvgPool → {tuple(y.shape)}')

---
## 🏛️ 13. A Complete CNN in PyTorch

We define a reusable `CNN` class using the structure:

```
features   → Conv + ReLU + Pool  (×2)
classifier → AdaptiveAvgPool + Flatten + Linear
```

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # → [N, 128, 1, 1]
            nn.Flatten(),                  # → [N, 128]
            nn.Linear(128, num_classes),   # → [N, num_classes]
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Sanity check
model = CNN(num_classes=10)
x = torch.randn(4, 3, 32, 32)
y = model(x)
print('Input :', x.shape)
print('Output:', y.shape)
print('Params:', sum(p.numel() for p in model.parameters()), 'total')

---
## 🏃 14. Training a CNN

The training loop is **unchanged** from previous modules:

```text
Image → CNN → Logits → CrossEntropyLoss → backward → optimizer.step()
```

We will now train on a real dataset.

---
## 📦 15. Loading a Real Dataset (CIFAR-10)

We'll use **CIFAR-10** — 60,000 small 32×32 color images across 10 classes.

We'll apply a **normalization** transform so the network sees well-conditioned inputs.

In [ ]:
# Transforms: ToTensor + Normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_ds = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

CLASSES = train_ds.classes
print('Classes:', CLASSES)
print('Train size:', len(train_ds))
print('Test  size:', len(test_ds))

In [ ]:
# Show a few sample images
def unnormalize(img):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    img, label = train_ds[i]
    ax.imshow(unnormalize(img).permute(1, 2, 0))
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🔁 16. Training & Evaluation Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        correct      += (logits.argmax(1) == yb).sum().item()
        total        += xb.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        running_loss += loss.item() * xb.size(0)
        correct      += (logits.argmax(1) == yb).sum().item()
        total        += xb.size(0)
    return running_loss / total, correct / total

print('Train / eval helpers ready.')

In [ ]:
# Initialize model, optimizer, loss
torch.manual_seed(42)

model = CNN(in_channels=3, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train for a few epochs
EPOCHS = 10
history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(model, test_loader, criterion)

    history['train_loss'].append(tr_loss); history['test_loss'].append(te_loss)
    history['train_acc'].append(tr_acc);   history['test_acc'].append(te_acc)

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
          f'test_loss={te_loss:.4f}   test_acc={te_acc:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['test_loss'],  label='test')
axes[0].set_title('Loss');     axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['test_acc'],  label='test')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 🔍 17. Inspecting Predictions

In [ ]:
# Predict on a small batch and visualize
model.eval()
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)

with torch.no_grad():
    logits = model(xb)
    probs  = F.softmax(logits, dim=1)
    preds  = logits.argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    img = xb[i].cpu()
    ax.imshow(unnormalize(img).permute(1, 2, 0))
    p = preds[i].item(); t = yb[i].item()
    color = 'green' if p == t else 'red'
    ax.set_title(f'P: {CLASSES[p]}\nT: {CLASSES[t]}', fontsize=8, color=color)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🎨 18. Visualizing Learned Filters

The first convolution's filters are 3×3 kernels we can visualize directly.
This gives us a peek at what low-level features the network learned.

In [ ]:
# Extract first conv layer weights
W = model.features[0].weight.detach().cpu()   # [32, 3, 3, 3]
print('First conv weight shape:', W.shape)

# Normalize each filter for display
def normalize_kernel(k):
    k = k - k.min()
    return k / (k.max() + 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    kernel = W[i].permute(1, 2, 0)          # [3, 3, 3] → HWC
    ax.imshow(normalize_kernel(kernel))
    ax.axis('off')
plt.suptitle('First-layer 3×3 filters (32 total)')
plt.tight_layout(); plt.show()

---
## 🚨 19. Common CNN Mistakes

### ❌ 1. Wrong tensor order
PyTorch expects `[N, C, H, W]`, **not** `[N, H, W, C]`.

### ❌ 2. Wrong `in_channels`
- Grayscale image → `in_channels=1`
- RGB image → `in_channels=3`

### ❌ 3. Hardcoded flattened size
After conv + pool, spatial dims shrink. Use `AdaptiveAvgPool2d((1,1))` or compute dynamically.

### ❌ 4. Softmax before CrossEntropyLoss
`nn.CrossEntropyLoss` **already includes** the softmax internally.
Pass raw logits directly:

```python
loss = nn.CrossEntropyLoss()(logits, labels)   # ✅
loss = nn.CrossEntropyLoss()(F.softmax(logits, 1), labels)  # ❌
```

### ❌ 5. Forgetting `model.eval()` at inference
Dropout and BatchNorm behave differently in eval mode.

---
## ⚠️ 20. CNN Limitations

CNNs are powerful, but not perfect:

- Require **lots of labeled data** for high-accuracy tasks
- Can be **computationally expensive**
- Architecture choices matter a lot
- Standard convolutions capture **local** patterns — long-range dependencies need deeper nets or attention

Modern vision systems increasingly combine CNNs with **attention** or use **Vision Transformers (ViTs)**.

### CNN vs Fully Connected

| Feature | Dense | CNN |
| --- | --- | --- |
| Spatial structure | Poor | Preserved |
| Local patterns | Not specialized | Excellent |
| Parameter efficiency | Low | High |
| Weight sharing | No | Yes |
| Translation invariance | Weak | Strong |

---
## 🏋️ 21. Hands-On Exercises

### Exercise 1 — Kernel Size Sweep
Train three CNNs with `kernel_size` = 3, 5, 7. Compare validation accuracy and parameter counts.

### Exercise 2 — Add Dropout
Add `nn.Dropout2d(p=0.25)` after each ReLU in the `features` block. Does test accuracy improve?

### Exercise 3 — BatchNorm
Insert `nn.BatchNorm2d` after each `Conv2d`. How do training curves change?

### Exercise 4 — Data Augmentation
Add `transforms.RandomHorizontalFlip()` and `transforms.RandomCrop(32, padding=4)` to the training transform. Retrain and compare.

### Exercise 5 — Filter Visualization
Visualize the **second** convolution's filters. Are they interpretable? Why or why not?

### Exercise 6 — Feature Maps
Pass a single image through `model.features` and visualize a few intermediate feature maps across channels.

### Exercise 7 — MNIST
Replace CIFAR-10 with MNIST (`in_channels=1`). Adjust the model accordingly and train.

---
## 🎤 22. Interview Questions

1. What is a CNN?
2. Why are CNNs useful for images?
3. What is convolution?
4. What is a kernel / filter?
5. What is a feature map?
6. What is stride?
7. What is padding?
8. How do you compute convolution output size?
9. What is parameter sharing?
10. What is local connectivity?
11. What is a receptive field?
12. What is pooling?
13. Max pooling vs average pooling?
14. Why do CNNs use multiple filters?
15. How do CNN parameters get learned?
16. What is the shape of a PyTorch image batch?
17. How do you compute Conv2d parameter count?
18. Why does spatial size decrease?
19. Why use adaptive pooling?
20. CNN vs fully-connected network?

---
## ✅ 23. Knowledge Check

- [ ] Explain why CNNs are useful
- [ ] Understand image tensor shapes
- [ ] Explain convolution
- [ ] Perform a simple convolution manually
- [ ] Explain kernels
- [ ] Explain feature maps
- [ ] Explain channels
- [ ] Explain stride
- [ ] Explain padding
- [ ] Calculate output dimensions
- [ ] Calculate convolution parameters
- [ ] Explain receptive fields
- [ ] Explain pooling
- [ ] Build a CNN with PyTorch
- [ ] Train an image classifier
- [ ] Debug tensor-shape problems
- [ ] Explain CNN limitations

---
## 🏁 24. Key Takeaways

The main CNN idea:

```text
Image
 ↓
Local filters
 ↓
Feature maps
 ↓
Learn visual patterns
 ↓
Combine simple patterns
 ↓
Recognize complex objects
```

### The Three Pillars of CNNs

1. **Local connectivity** — filters look at local regions
2. **Parameter sharing** — the same filter is reused across the image
3. **Hierarchical features** — edges → textures → shapes → objects

### 🧭 The Golden Rule

> **Don't flatten images into huge dense layers — let convolutions learn spatial structure.**